In [1]:
import numpy as np
import os
import pandas as pd
from sqlalchemy import create_engine, text
from glob import glob

# Database connections
engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()
engine = create_engine("sqlite:///c:\\ruby\\portmy\\db\\development.sqlite3")
conmy = engine.connect()

In [3]:
# Get the user's home directory
user_path = os.path.expanduser('~')
# Get the current working directory
current_path = os.getcwd()
# Derive the base directory (base_dir) by removing the last folder ('Daily')
base_path = os.path.dirname(current_path)
#C:\Users\PC1\OneDrive\A5\Data
dts_path = os.path.join(user_path, "OneDrive", "Downloads", "Datasets")
print(f"Datasets path: {dts_path}")
dat_path = os.path.join(base_path, "Data")
#C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data>
god_path = os.path.join(user_path, "OneDrive","Imports","santisoontarinka@gmail.com - Google Drive","Data")
#C:\Users\PC1\iCloudDrive\data
icd_path = os.path.join(user_path, "iCloudDrive", "Data")
#C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data
osd_path = os.path.join(user_path, "OneDrive","Documents","obsidian-git-sync","Data")

Datasets path: C:\Users\PC1\OneDrive\Downloads\Datasets


In [5]:
print("User path:", user_path)
print(f"Current path: {current_path}")
print(f"Base path: {base_path}")
print(f"Datasets path : {dts_path}") 
print(f"Data path : {dat_path}") 
print(f"Google Drive path : {god_path}")
print(f"iCloudDrive path : {icd_path}") 
print(f"Obsidian path : {osd_path}")

User path: C:\Users\PC1
Current path: C:\Users\PC1\OneDrive\A4\Data Cleansing
Base path: C:\Users\PC1\OneDrive\A4
Datasets path : C:\Users\PC1\OneDrive\Downloads\Datasets
Data path : C:\Users\PC1\OneDrive\A4\Data
Google Drive path : C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data
iCloudDrive path : C:\Users\PC1\iCloudDrive\Data
Obsidian path : C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data


In [7]:
sql = f"""
SELECT *
FROM tickers
ORDER BY name
"""
print(sql)
df_tickers_lt = pd.read_sql(sql, conlt)
print(df_tickers_lt.shape)
print(df_tickers_lt.dtypes)


SELECT *
FROM tickers
ORDER BY name

(390, 9)
id             int64
name          object
full_name     object
sector        object
subsector     object
market        object
website       object
created_at    object
updated_at    object
dtype: object


In [9]:
sql = f"""
SELECT *
FROM tickers
ORDER BY name
"""
print(sql)
df_tickers_my = pd.read_sql(sql, conmy)
print(df_tickers_my.shape)
print(df_tickers_my.dtypes)


SELECT *
FROM tickers
ORDER BY name

(396, 9)
id             int64
name          object
full_name     object
sector        object
subsector     object
market        object
website       object
created_at    object
updated_at    object
dtype: object


In [11]:
tickers_lt_set = set(df_tickers_lt['name'].to_list())
tickers_my_set = set(df_tickers_my['name'].to_list())
lt_my_dif_set = tickers_lt_set - tickers_my_set
lt_my_dif_set

{'AURA', 'BTG', 'MOSHI', 'STECON', 'TLI'}

In [13]:
# --- ADD MISSING TICKERS FROM portlt TO portmy (SQLite) ---

print("\n" + "="*80)
print("SYNC TICKERS: portlt (SQLite) → portmy (SQLite)")
print("="*80)

# 1. Identify tickers present in portlt but missing in portmy
tickers_lt_set = set(df_tickers_lt['name'])
tickers_my_set = set(df_tickers_my['name'])
missing_tickers = tickers_lt_set - tickers_my_set

if not missing_tickers:
    print("✅ No missing tickers – portmy is already in sync with portlt.")
else:
    print(f"📌 Missing tickers in portmy: {len(missing_tickers)}")
    print(f"   {sorted(missing_tickers)}")

    # 2. Fetch full records from portlt (source)
    placeholders = ','.join(['?'] * len(missing_tickers))
    sql = f"""
        SELECT name, full_name, sector, subsector, market, website, created_at, updated_at
        FROM tickers
        WHERE name IN ({placeholders})
    """
    df_missing = pd.read_sql(sql, conlt, params=tuple(missing_tickers))
    print(f"📥 Retrieved {len(df_missing)} records from portlt.")

    # 3. Insert into portmy – drop the 'id' column so SQLite auto-generates it
    try:
        df_missing.to_sql('tickers', conmy, if_exists='append', index=False)
        print(f"✅ Successfully added {len(df_missing)} tickers to portmy.")
    except Exception as e:
        # If a duplicate key occurs (rare if we computed missing correctly), fallback to INSERT OR IGNORE
        if "UNIQUE constraint failed" in str(e) or "duplicate" in str(e).lower():
            print("⚠️  Some tickers already exist – using INSERT OR IGNORE to skip duplicates.")
            # Use raw SQLite INSERT OR IGNORE for safety
            with conmy.engine.begin() as conn:
                for _, row in df_missing.iterrows():
                    conn.execute(
                        text("""
                            INSERT OR IGNORE INTO tickers 
                            (name, full_name, sector, subsector, market, website, created_at, updated_at)
                            VALUES (:name, :full_name, :sector, :subsector, :market, :website, :created_at, :updated_at)
                        """),
                        row.to_dict()
                    )
            print(f"✅ Inserted/ignored {len(df_missing)} records.")
        else:
            print(f"❌ Error: {e}")
            raise

# 4. Verification
df_tickers_my_after = pd.read_sql("SELECT name FROM tickers ORDER BY name", conmy)
tickers_my_after_set = set(df_tickers_my_after['name'].tolist())
still_missing = missing_tickers - tickers_my_after_set

print("\n" + "-"*60)
print("VERIFICATION")
print("-"*60)
print(f"Tickers in portlt: {len(tickers_lt_set)}")
print(f"Tickers in portmy after sync: {len(tickers_my_after_set)}")
if not still_missing:
    print("✅ SUCCESS: All tickers from portlt are now present in portmy.")
else:
    print(f"⚠️  WARNING: Still missing {len(still_missing)} tickers: {sorted(still_missing)}")
print("="*80)


SYNC TICKERS: portlt (SQLite) → portmy (SQLite)
📌 Missing tickers in portmy: 5
   ['AURA', 'BTG', 'MOSHI', 'STECON', 'TLI']
📥 Retrieved 5 records from portlt.
✅ Successfully added 5 tickers to portmy.

------------------------------------------------------------
VERIFICATION
------------------------------------------------------------
Tickers in portlt: 390
Tickers in portmy after sync: 401
✅ SUCCESS: All tickers from portlt are now present in portmy.
